# Model comparison

EnvPG uses its learned layout; IPPO, MAPPO and no yaw use the reference layout. Every final score uses the same WFCRL reward, 25 wind bins and 150 controlled steps per bin. Critics are not used for scoring. Training budgets differ.


In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = Path.cwd().resolve()
assert (ROOT / "envpg.py").exists(), "Open from the repository root"
os.environ.setdefault("MPLCONFIGDIR", str(ROOT / "__simul__" / "matplotlib"))
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT / "wfcrl-benchmark" / "algos"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import yaml
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator
from floris.tools.visualization import plot_turbines_with_fi, visualize_cut_plane
from envpg import Actor, REFERENCE, windrose
from wfcrl import environments as envs
from extractors import VectorExtractor
from baseline_ippo import Agent as IPPOAgent
from baseline_mappo import Agent as MAPPOAgent

RUNS = {
    "EnvPG": ROOT / "runs/envpg-001",
    "IPPO": ROOT / "wfcrl-benchmark/runs/Dec_Ablaincourt_Floris__baseline_ippo__windrose_1__1794149416",
    "MAPPO": ROOT / "wfcrl-benchmark/runs/Dec_Ablaincourt_Floris__baseline_mappo__windrose_1__1794150208",
}
ORDER = ("No yaw", *RUNS)
for name, path in RUNS.items():
    assert path.is_dir(), f"Missing {name}: {path}"
config = json.loads((RUNS["EnvPG"] / "config.json").read_text())["parameters"]
H = config["horizon"]
WIND_DATA = ROOT / "wfcrl-benchmark/data/smarteole.csv"
assert Path(config["wind_data"]).name == WIND_DATA.name
for name in ("IPPO", "MAPPO"):
    saved = yaml.load((RUNS[name] / "config.yaml").read_text(), Loader=yaml.BaseLoader)
    assert saved["env_id"] == "Dec_Ablaincourt_Floris"
    assert saved["scenario"] == "windrose"
    assert int(saved["episode_length"]) == H + 1
    assert float(saved["load_coef"]) == config["load_coef"]
    assert Path(saved["wind_data"]).name == WIND_DATA.name
print(f"{H} steps/bin, load coefficient {config['load_coef']}")


## Training

The curves use their own training axes. EnvPG's actor quantity is a policy-gradient surrogate; its height is not a reward or a measure of progress.


In [ ]:
events = EventAccumulator(str(RUNS["EnvPG"] / "tb"))
events.Reload()

def scalar(tag):
    return pd.DataFrame([(e.step, e.value) for e in events.Scalars(tag)],
                        columns=("step", "value"))

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
curve = scalar("train/reward_mean")
ax[0].plot(curve.step, curve.value)
ax[0].set(title="EnvPG training", xlabel="Design iteration", ylabel="Reward / step")
for name in ("IPPO", "MAPPO"):
    farm = pd.read_csv(RUNS[name] / "farm.csv")
    ax[1].plot(farm["index"], farm.episode_reward / H, label=name)
ax[1].set(title="WFCRL training", xlabel="Environment steps", ylabel="Reward / step")
ax[1].legend()
curve = scalar("eval/windrose_score")
ax[2].plot(curve.step, curve.value)
ax[2].set(title="EnvPG windrose evaluation", xlabel="Design iteration", ylabel="Weighted reward")
for a in ax:
    a.grid(alpha=.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for a, tag, title in zip(ax,
                         ("loss/actor", "loss/critic", "loss/cost_sensitivity"),
                         ("Actor surrogate", "Critic TD loss", "Cost-sensitivity loss")):
    curve = scalar(tag)
    a.plot(curve.step, curve.value)
    a.set(title=title, xlabel="Design iteration")
    a.grid(alpha=.2)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
for name in ("IPPO", "MAPPO"):
    losses = pd.read_csv(RUNS[name] / "losses.csv")
    for a, suffix in zip(ax, ("policy_loss", "value_loss")):
        columns = [c for c in losses if c.endswith(suffix)]
        if columns:
            a.plot(losses["index"], losses[columns].mean(axis=1), label=name)
for a, title in zip(ax, ("PPO policy loss", "PPO value loss")):
    a.set(title=title, xlabel="Environment steps")
    a.legend()
    a.grid(alpha=.2)
plt.tight_layout()
plt.show()


## Final checkpoints

Each policy is run in the same multi-agent FLORIS environment with `load_coef=0.1`. No yaw gives a reference score without a learned controller.


In [ ]:
def load_model(name):
    env = envs.make("Dec_Ablaincourt_Floris", controls=["yaw"], log=False,
                    max_num_steps=H + 1, load_coef=config["load_coef"])
    model = {"env": env, "layout": REFERENCE.copy()}
    if name == "No yaw":
        return model
    if name == "EnvPG":
        # The local checkpoint stores its layout as a NumPy array.
        state = torch.load(RUNS[name] / "latest.pt", map_location="cpu", weights_only=False)
        actor = Actor(H, config["max_displacement"]).eval()
        actor.load_state_dict(state["actor"])
        model.update(actor=actor, z=np.asarray(state["layout"]))
        model["layout"][1:] += model["z"].reshape(6, 2)
        env.mdp.interface.fi.reinitialize(layout_x=model["layout"][:, 0],
                                          layout_y=model["layout"][:, 1])
        return model

    saved = yaml.load((RUNS[name] / "config.yaml").read_text(), Loader=yaml.BaseLoader)
    first = env.possible_agents[0]
    observation = VectorExtractor(env.observation_space(first))
    action = VectorExtractor(env.action_space(first))
    cls = IPPOAgent if name == "IPPO" else MAPPOAgent
    agents = []
    for i in range(env.num_turbines):
        agent = cls(observation.space, action.space,
                    tuple(map(int, saved["hidden_layer_nn"]))).eval()
        path = RUNS[name] / f"baseline_{name.lower()}.cleanrl_model_{i}"
        agent.load_state_dict(torch.load(path, map_location="cpu", weights_only=True))
        agents.append(agent)
    model.update(agents=agents, observation=observation, action=action)
    return model

models = {name: load_model(name) for name in ORDER}


In [ ]:
def command_for(name, model, turbine, index, joint_action):
    if name == "No yaw":
        return {"yaw": np.zeros(1, dtype=np.float32)}
    if name == "EnvPG":
        return {"yaw": np.array([joint_action[index]], dtype=np.float32)}
    obs = torch.tensor(model["observation"](model["env"].observe(turbine)),
                       dtype=torch.float32)
    agent = model["agents"][index]
    if name == "IPPO":
        action = agent.get_action_and_value(obs, deterministic=True)[0]
    else:
        action = agent.get_action(obs, deterministic=True)[0]
    return model["action"].make_dict(action)

def run_case(name, direction, speed):
    model = models[name]
    env = model["env"]
    env.reset(options={"wind_speed": float(speed), "wind_direction": float(direction)})
    first = env.possible_agents[0]
    rows = []
    with torch.no_grad():
        for _ in range(H):
            joint_action = None
            if name == "EnvPG":
                usage = [env.accumulated_actions[t]["yaw"] for t in env.possible_agents]
                state = np.r_[env._state["freewind_measurements"], env._state["yaw"],
                              usage, env.num_moves]
                joint_action = model["actor"].sample(
                    torch.tensor(state, dtype=torch.float32)[None],
                    torch.tensor(model["z"], dtype=torch.float32)[None],
                    deterministic=True)[0].numpy()
            for i, turbine in enumerate(env.possible_agents):
                env.step(command_for(name, model, turbine, i, joint_action))
            rows.append({"reward": float(env.rewards[first].item()),
                         "power_mw": sum(env.infos[t]["power"] for t in env.possible_agents),
                         "load_proxy": np.mean(np.abs([env.infos[t]["load"]
                                                       for t in env.possible_agents])),
                         "yaw": np.asarray(env._state["yaw"]).copy()})
    assert env.truncations[first], f"{name} did not complete {H} steps"
    return rows


In [ ]:
conditions = windrose(WIND_DATA)
assert len(conditions) == 25
assert abs(sum(weight for _, _, weight in conditions) - 1) < 1e-9

scores = []
for name in ORDER:
    for direction, speed, weight in conditions:
        rows = run_case(name, direction, speed)
        scores.append({"model": name, "weight": weight,
                       "reward": sum(r["reward"] for r in rows),
                       "power_mw": np.mean([r["power_mw"] for r in rows]),
                       "load_proxy": np.mean([r["load_proxy"] for r in rows])})
frame = pd.DataFrame(scores)
metrics = ("reward", "power_mw", "load_proxy")
summary = pd.DataFrame({key: (frame[key] * frame.weight).groupby(frame.model,
                            sort=False).sum() for key in metrics})
summary["reward_vs_no_yaw"] = summary.reward - summary.loc["No yaw", "reward"]
display(summary.round(3))

fig, ax = plt.subplots(1, 3, figsize=(12, 3))
for a, key, title in zip(ax, metrics, ("Weighted reward / 150 steps",
                                       "Farm power (MW)", "Load proxy")):
    a.bar(summary.index, summary[key])
    a.set_title(title)
    a.grid(axis="y", alpha=.2)
plt.tight_layout()
plt.show()

initial = scalar("eval/windrose_score").value.iloc[0]
final = scalar("eval/windrose_score").value.iloc[-1]
shift = np.linalg.norm(models["EnvPG"]["layout"][1:] - REFERENCE[1:], axis=1).max()
print(f"EnvPG logged evaluation: {initial:.1f} → {final:.1f}; largest turbine shift: {shift:.2f} m")


## One wind condition

The traces and hub-height wakes use 8 m/s wind from 270° after 150 control steps. The score table above uses all 25 wind bins.


In [ ]:
example = {name: run_case(name, 270.0, 8.0) for name in ORDER}
fig, ax = plt.subplots(1, 3, figsize=(13, 3))
for name, rows in example.items():
    for a, key in zip(ax, ("reward", "power_mw", "load_proxy")):
        a.plot([r[key] for r in rows], label=name)
for a, title in zip(ax, ("Reward / step", "Farm power (MW)", "Load proxy")):
    a.set(title=title, xlabel="Control step")
    a.grid(alpha=.2)
ax[0].legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(1, 4, figsize=(14, 3))
for a, (name, rows) in zip(ax, example.items()):
    yaw = np.stack([r["yaw"] for r in rows]).T
    image = a.imshow(yaw, aspect="auto", origin="lower", cmap="coolwarm",
                     vmin=-40, vmax=40)
    a.set(title=f"{name} yaw (°)", xlabel="Control step", ylabel="Turbine")
    a.set_yticks(range(7), range(1, 8))
fig.colorbar(image, ax=ax, label="Yaw (°)")
plt.show()


In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(11, 12), sharex=True, sharey=True)
for a, name in zip(ax, ORDER):
    fi = models[name]["env"].mdp.interface.fi
    yaw = fi.floris.farm.yaw_angles.copy()
    plane = fi.calculate_horizontal_plane(height=90, x_resolution=100, y_resolution=50,
                                          x_bounds=(200, 2800), y_bounds=(-600, 600),
                                          wd=[270.0], ws=[8.0], yaw_angles=yaw)
    image = visualize_cut_plane(plane, ax=a, min_speed=0, max_speed=8,
                                levels=np.linspace(0, 8, 21))
    plot_turbines_with_fi(fi, ax=a, wd=np.array([270.0]), yaw_angles=yaw)
    for i, (x, y) in enumerate(models[name]["layout"]):
        a.text(x, y + 65, str(i + 1), ha="center", fontsize=8)
    a.set(title=name, ylabel="y (m)")
ax[-1].set_xlabel("x (m)")
fig.colorbar(image, ax=ax, label="Wind speed (m/s)")
plt.show()
for model in models.values():
    model["env"].close()
